In [1]:
!apt-get update
!apt-get install -y libgl1-mesa-glx


Hit:1 http://deb.debian.org/debian bookworm InRelease
Hit:2 http://deb.debian.org/debian bookworm-updates InRelease
Hit:3 http://deb.debian.org/debian-security bookworm-security InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1-mesa-glx is already the newest version (22.3.6-1+deb12u1).
0 upgraded, 0 newly installed, 0 to remove and 43 not upgraded.


In [2]:
!pip install mediapipe opencv-python tensorflow

INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of tensorflow to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 1.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 1.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 1.6 MB/s eta 0:00:0000:0100:10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 1.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 M

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf


2025-11-17 19:12:54.215581: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-17 19:12:54.478071: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-17 19:12:54.723008: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763406776.320008     174 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763406776.636933     174 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763406777.200734     174 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [4]:
facenet = tf.keras.applications.InceptionResNetV2(
    include_top=False,
    pooling='avg',
    input_shape=(160,160,3)
)


2025-11-17 19:13:02.802023: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 121s 1us/step


In [5]:
mp_face = mp.solutions.face_detection
detector = mp_face.FaceDetection(model_selection=1, min_detection_confidence=0.5)

def extract_face(img):
    results = detector.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if not results.detections:
        return None

    detection = results.detections[0]
    box = detection.location_data.relative_bounding_box
    h, w, _ = img.shape

    x1 = int(box.xmin * w)
    y1 = int(box.ymin * h)
    x2 = x1 + int(box.width * w)
    y2 = y1 + int(box.height * h)

    face = img[y1:y2, x1:x2]
    face = cv2.resize(face, (160,160))
    return face


In [6]:
known_embeddings = []
known_names = []

def get_embedding(img):
    img = img.astype(np.float32)
    img = (img - 127.5) / 128.0
    return facenet.predict(img[np.newaxis, ...])[0]

# Example: Saifur
img = cv2.imread("known/saifur.jpg")
face = extract_face(img)
emb = get_embedding(face)

known_embeddings.append(emb)
known_names.append("Saifur Rahman")


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1763406907.895216     818 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


In [7]:
import cv2

cam_index = 0  # Try 0, 1, 2... depending on your system
cap = cv2.VideoCapture(cam_index)

# Check if camera opened
if not cap.isOpened():
    print("❌ Camera not found!")
else:
    print("✅ Camera opened successfully")

    # Try reading a frame
    ret, frame = cap.read()
    if not ret:
        print("⚠️ Camera opened but failed to read frame!")
    else:
        print("📷 Camera is working and frame captured.")

cap.release()


✅ Camera opened successfully
📷 Camera is working and frame captured.


In [ ]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()

    face = extract_face(frame)

    if face is not None:
        emb = get_embedding(face)

        distances = [np.linalg.norm(emb - k) for k in known_embeddings]
        idx = np.argmin(distances)

        if distances[idx] < 0.9:
            name = known_names[idx]
        else:
            name = "Unknown"

        cv2.putText(frame, name, (30, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Face Recognition (No dlib)", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
